# CEREBRONET: Brain Tumor Inference and Explainability Demo

This notebook demonstrates how to load a trained **CerebroNetCNN** model checkpoint, run classification inference on a brain MRI scan, and visualize class-activation heatmaps using our from-scratch **Grad-CAM** implementation.

In [ ]:
import os
import sys
import torch
import matplotlib.pyplot as plt

# Ensure the src directory is in the path
sys.path.append(os.path.abspath('..'))

from src.config import load_config
from src.dataset import CLASSES, BrainMRIDataset
from src.transforms import get_val_test_transforms
from src.model import CerebroNetCNN, OncoNetCNN
from src.gradcam import GradCAM, overlay_heatmap, get_last_conv_layer
from src.visualization import plot_gradcam_grid, plot_feature_maps, plot_learned_filters

### 1. Load Configurations and Initialize Device

In [ ]:
config = load_config('../configs/base_config.yaml')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

### 2. Restore Trained Model Checkpoint

In [ ]:
model = CerebroNetCNN(num_classes=len(CLASSES))
checkpoint_path = '../checkpoints/custom_best.pth'

if os.path.exists(checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    print("Trained model checkpoint successfully restored!")
else:
    print(f"Trained model not found at {checkpoint_path}. Please train a model first.")

model = model.to(device)
model.eval()

### 3. Load Sample Test Image

In [ ]:
test_transform = get_val_test_transforms(img_size=config.data.img_size)
test_dataset = BrainMRIDataset('../data/processed/test', transform=test_transform)

# Select a sample from the test set
img_tensor, label, orig_img = test_dataset[0]
print(f"Original label: {CLASSES[label]} (Index: {label})")

plt.figure(figsize=(5,5))
plt.imshow(orig_img)
plt.title(f"True Category: {CLASSES[label]}")
plt.axis('off')
plt.show()

### 4. Run Inference and Apply Grad-CAM

In [ ]:
# Add batch dimension and load input image tensor
input_tensor = img_tensor.unsqueeze(0).to(device)
input_tensor.requires_grad = True

# 1. Forward pass
outputs = model(input_tensor)
probs = torch.softmax(outputs, dim=1)[0]
pred_idx = torch.argmax(probs).item()
confidence = probs[pred_idx].item()

print(f"Predicted Class: {CLASSES[pred_idx].upper()}")
print(f"Confidence Score: {confidence * 100:.2f}%")

# 2. Resolve target layer and compute Grad-CAM
target_layer = get_last_conv_layer('custom', model)

with GradCAM(model, target_layer) as gcam:
    heatmap = gcam.generate_heatmap(input_tensor, target_class=pred_idx)

# 3. Overlay transparent activation colormap
colored_heatmap, blended = overlay_heatmap(heatmap, orig_img, alpha=0.4)

# 4. Plot explainability grid
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(orig_img)
axes[0].set_title("Original MRI Scan")
axes[0].axis('off')

axes[1].imshow(heatmap, cmap='jet')
axes[1].set_title("Grad-CAM Feature Heatmap")
axes[1].axis('off')

axes[2].imshow(blended)
axes[2].set_title(f"Overlay - Pred: {CLASSES[pred_idx]} ({confidence*100:.1f}%)")
axes[2].axis('off')

plt.tight_layout()
plt.show()

### 5. Intermediate Features Visualization

In [ ]:
# Plot learned filters of the first layer
plt.figure(figsize=(10,10))
plot_learned_filters(model, save_path='sample_filters.png')
plt.close() # visualised inline or from saved file

# Plot intermediate activation feature maps
plot_feature_maps(model, input_tensor, save_path='sample_activation_maps.png')